# Introduction to Stream Processing

Every page so far has processed **batches**: a query reads all of a table,
computes a result and stops. The pipeline in
{doc}`ETL with the DataFrame API <../01_introduction_to_apache_spark/04_etl_with_the_dataframe_api/index>`
rebuilt its tables from scratch on every run. That works while data arrives
in daily dumps and a day-old answer is good enough. Many questions cannot
wait that long, though: is this card payment fraudulent, is this machine
overheating, how many pastries has each bakery sold in the last ten minutes?
This notebook introduces the ideas behind **stream processing**. The rest of
the chapter shows how Spark implements them.

## Bounded and Unbounded Data

A **bounded** dataset has a known end: yesterday's sales, last month's logs.
You can read all of it before computing anything. An **unbounded** dataset
never ends. New sales, clicks or sensor readings keep arriving for as long as
the business runs, so there is never a moment when "all the data" is
available.

**Batch processing** handles bounded data. **Stream processing** handles
unbounded data, computing results continuously and updating them as new
records arrive.

```mermaid
:caption: A batch job runs over a complete, fixed input. A streaming job runs over an input that keeps growing, updating its result as new records arrive.
:align: center

graph LR
    subgraph Batch
        B1[("Complete dataset")] --> B2["Query runs once"] --> B3[("Result")]
    end
    subgraph Streaming
        S1(["New records keep arriving"]) --> S2["Query runs continuously"] --> S3[("Result kept up to date")]
    end
```

The difference is less about the size of the data than about **when** you
need the answer. Streaming trades some simplicity for lower **latency**, the
time between an event happening and its effect showing up in a result.

## Why Stream?

- **Freshness.** Dashboards, alerts and fraud checks need results within
  seconds or minutes, not after a nightly job.
- **Incremental work.** A batch job that reprocesses a growing table on every
  run does more work each day. A streaming job computes its result
  incrementally
  ([Structured Streaming concepts](https://docs.databricks.com/aws/en/structured-streaming/concepts)):
  it remembers how far it got and on the next run only processes what is
  new. This makes streaming attractive even when latency does not matter.
  For append-only sources, Databricks recommends streaming reads from the
  bronze layer of a medallion pipeline
  ([medallion: silver](https://docs.databricks.com/aws/en/lakehouse/medallion#build-silver-tables-from-the-bronze-layer)).
- **Even load.** Processing a little data often spreads work over the day
  instead of concentrating it in one large nightly window.

The cost is complexity. A streaming job has to remember what it has
processed, cope with records that arrive late or twice, and keep running
correctly through failures and restarts.

## Event Time and Processing Time

Every record in a stream has two timestamps that are easy to confuse:

- **Event time** is when the event actually happened, for example when the
  customer paid. It is the time embedded in the data itself
  ([Apache Spark: Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/streaming/getting-started.html#handling-event-time-and-late-data)),
  such as `dateTime` in `samples.bakehouse.sales_transactions`.
- **Processing time** is when the stream processor sees the record.

In a perfect world they would be nearly equal. In practice the gap varies:
a payment terminal loses its connection and uploads an hour of sales at
once, a mobile app buffers events until the phone is back online, or an
upstream system restarts. Records therefore arrive **out of order**, and some
arrive **late**, after results for their time period have already been
computed.

(stream-out-of-order)=
:::::{dropdown} Example: Out-of-Order Arrival
:open:
:color: primary
:icon: pencil
:class-body: example-body

Three sales happen at a bakery, and the stream processor receives them in
this order:

```text
arrival order  event_time  processing_time
1              10:02       10:02
2              10:11       10:11
3              10:06       10:15   <- the terminal was offline for 9 minutes
```

**Group by processing time** into 10-minute buckets. Sale 1 lands in the
10:00 bucket, and sales 2 and 3 land in the 10:10 bucket:

```text
10:00-10:10  1 sale
10:10-10:20  2 sales
```

**Group by event time** instead. Sales 1 and 3 both happened between 10:00
and 10:10:

```text
10:00-10:10  2 sales
10:10-10:20  1 sale
```

Only the event-time result describes what happened in the shop. The price is
that the 10:00 bucket cannot be considered final at 10:10, because sales from
that period can still arrive. Deciding how long to wait for them is the job
of the *watermark*, introduced in
{doc}`Stream Aggregations, Windows and Watermarks <03_stream_aggregations_windows_and_watermarks>`.
:::::

## Processing Models

Stream processors execute in one of two broad ways:

- **Record at a time.** Each record flows through the pipeline as soon as it
  arrives. This gives the lowest latency, down to milliseconds, but makes
  fault tolerance and high throughput harder to achieve.
- **Micro-batch.** The processor collects whatever arrived since the last
  step into a small batch and processes it with an ordinary batch engine.
  Each step adds some scheduling overhead, so latency is typically hundreds
  of milliseconds to seconds or more, but throughput is high and recovery is
  simple: a failed micro-batch is just run again.

Spark's **Structured Streaming** uses a micro-batch engine by default, with
end-to-end latencies as low as 100 milliseconds in open-source Spark
([Apache Spark: Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/streaming/index.html)).
On Databricks, **real-time mode** processes data as it arrives instead, with
end-to-end latency as low as five milliseconds
([Real-time mode in Structured Streaming](https://docs.databricks.com/aws/en/structured-streaming/real-time#what-is-real-time-mode)). It is covered in
{doc}`Real-Time Mode in Structured Streaming <04_real_time_mode_in_structured_streaming>`.

## Delivery Guarantees

When machines fail and jobs restart, a stream processor can lose records or
process them twice. Systems describe what they promise with three standard
terms:

| Guarantee | Meaning | Risk |
| --- | --- | --- |
| At most once | Each record is processed zero or one times | Data loss |
| At least once | Each record is processed one or more times | Duplicates |
| Exactly once | Each record affects the result exactly once | None, but needs cooperation from the source and the sink |

Structured Streaming provides end-to-end **exactly-once** semantics by
combining replayable sources, whose read position is tracked with offsets,
and idempotent sinks. The engine records the offset range of the data
processed in each trigger with checkpointing and write-ahead logs
([Apache Spark: Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/streaming/getting-started.html#fault-tolerance-semantics)).
On Databricks, the **checkpoint** stores the processing state that enables
fault tolerance and exactly-once delivery
([Structured Streaming concepts](https://docs.databricks.com/aws/en/structured-streaming/concepts)),
and Delta tables work as streaming sources and sinks with exactly-once
guarantees
([Streaming on Databricks](https://docs.databricks.com/aws/en/structured-streaming/)).
{doc}`Spark Structured Streaming <02_spark_structured_streaming>` shows the
checkpoint in action.

## Streaming on Databricks

On Databricks, streaming data usually enters through a message bus, or as
files landing in cloud storage
([Streaming on Databricks](https://docs.databricks.com/aws/en/structured-streaming/)).
Several tools build on Structured Streaming:

- **Structured Streaming** itself, the DataFrame API for streams, used
  throughout this chapter.
- **Auto Loader**, which provides a Structured Streaming source called
  `cloudFiles` that incrementally processes new data files as they arrive in
  cloud storage
  ([What is Auto Loader?](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/)).
- **Lakeflow pipelines**, which extend Apache Spark Declarative Pipelines,
  a declarative framework for building batch and streaming pipelines
  ([Spark Declarative Pipelines](https://docs.databricks.com/aws/en/ldp/)).

How often a streaming query runs is set by its **trigger**, which places it
somewhere on a spectrum of latency and cost:

| Trigger | Typical latency | Compute |
| --- | --- | --- |
| `availableNow`, scheduled as a job | Minutes to hours | Runs only while there is data to process |
| `processingTime`, micro-batches on an interval | Seconds to minutes | Always running |
| `realTime`, real-time mode | Milliseconds | Always running, classic compute only |

On serverless compute, which this course uses through Databricks Connect,
Databricks recommends the `availableNow` trigger; `processingTime` and
continuous triggers are not supported
([Serverless compute limitations](https://docs.databricks.com/aws/en/compute/serverless/limitations#streaming-limitations)),
and real-time mode needs classic compute
([Set up real-time mode](https://docs.databricks.com/aws/en/structured-streaming/real-time/setup#prerequisites)).
That suits learning well: each query processes everything new and then
stops, so a notebook cell finishes like any other.

## A First Look

A streaming DataFrame is created with `spark.readStream` instead of
`spark.read`. It looks like any other DataFrame and supports most of the
same transformations, but it describes data that never ends:

In [1]:
from project.utils.sparksession import get_spark_session

spark = get_spark_session(profile="free-edition")

stream = spark.readStream.table("samples.bakehouse.sales_transactions")

stream.isStreaming

True

Because the input has no end, actions that need *all* the data, such as
`show`, `count` or `collect`, make no sense and are rejected:

In [2]:
stream.show()

AnalysisException: Queries with streaming sources must be executed with writeStream.start(), or from a streaming table or flow definition within a Lakeflow Declarative Pipeline.;
tahoe

JVM stacktrace:
org.apache.spark.sql.catalyst.ExtendedAnalysisException
	at org.apache.spark.sql.catalyst.analysis.UnsupportedOperationChecker$.throwError(UnsupportedOperationChecker.scala:789)
	at org.apache.spark.sql.catalyst.analysis.UnsupportedOperationChecker$.$anonfun$checkForBatch$2(UnsupportedOperationChecker.scala:82)
	at org.apache.spark.sql.catalyst.analysis.UnsupportedOperationChecker$.$anonfun$checkForBatch$2$adapted(UnsupportedOperationChecker.scala:79)
	at org.apache.spark.sql.catalyst.trees.TreeNode.foreachUp(TreeNode.scala:406)
	at org.apache.spark.sql.catalyst.trees.TreeNode.$anonfun$foreachUp$1(TreeNode.scala:405)
	at org.apache.spark.sql.catalyst.trees.TreeNode.$anonfun$foreachUp$1$adapted(TreeNode.scala:405)
	at scala.collection.immutable.Vector.foreach(Vector.scala:2125)
	at org.ap

The `AnalysisException` says that queries with streaming sources must be
executed with `writeStream.start()`.

A streaming query has to be started with `writeStream`, which names a
destination and a **checkpoint location** where Spark records its progress.
The checkpoint lives in a Unity Catalog volume. The setup below creates a
volume for this notebook and removes the tables and checkpoints of any
earlier run, so the notebook can be run again from the top:

In [3]:
spark.sql("DROP VOLUME IF EXISTS workspace.default.streaming_01")
spark.sql("CREATE VOLUME workspace.default.streaming_01")
spark.sql("DROP TABLE IF EXISTS workspace.default.stream01_sales_copy")

CHECKPOINT = "/Volumes/workspace/default/streaming_01/checkpoints/sales_copy"

Now copy the transactions table into a new table, as a stream. The
`availableNow` trigger processes all data available right now and then
stops, and `awaitTermination` waits for that to happen:

In [4]:
def copy_new_sales():
    query = (
        spark.readStream.table("samples.bakehouse.sales_transactions")
        .writeStream
        .option("checkpointLocation", CHECKPOINT)
        .trigger(availableNow=True)
        .toTable("workspace.default.stream01_sales_copy")
    )
    query.awaitTermination()
    return sum(p["sources"][0]["numInputRows"] for p in query.recentProgress)


copy_new_sales()

3333

The first run processed every row of the source table. Run exactly the same
query again:

In [5]:
copy_new_sales()

0

This time it processed nothing, because the checkpoint records that all
existing data has been handled and the source has not changed since. Had new
sales been appended to the source table in between, the second run would
have processed only those. This is the incremental processing described
above, and the target table holds every sale exactly once:

In [6]:
spark.read.table("workspace.default.stream01_sales_copy").count()

3333

## When to Stream, and When to Run a Batch

| Situation | Choose |
| --- | --- |
| The source is rebuilt or replaced as a whole | Batch |
| Data keeps arriving and is only appended | Streaming with `availableNow`, scheduled as often as freshness requires |
| Results are needed within seconds | Streaming with a `processingTime` trigger on classic compute |
| Operational decisions within milliseconds, such as blocking fraud | Real-time mode |

Databricks positions real-time mode for sub-second, operational workloads,
and micro-batch for ETL, medallion pipelines and cost-sensitive workloads
([Real-time mode in Structured Streaming](https://docs.databricks.com/aws/en/structured-streaming/real-time#when-to-use-real-time-mode)).

## Common Mistakes

- **Starting a stream without a trigger on serverless compute.** Spark's
  default trigger is `ProcessingTime("0 seconds")`, which serverless doesn't
  support, so the query fails with
  `INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED`
  ([Serverless compute limitations](https://docs.databricks.com/aws/en/compute/serverless/limitations#streaming-limitations)).
  Always set `.trigger(availableNow=True)` there:

In [7]:
(
    spark.readStream.table("samples.bakehouse.sales_transactions")
    .writeStream
    .option("checkpointLocation", CHECKPOINT + "_no_trigger")
    .toTable("workspace.default.stream01_sales_copy")
)

AnalysisException: [INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED] Trigger type ProcessingTime is not supported for this cluster type.
Use a different trigger type e.g. AvailableNow, Once. SQLSTATE: 0A000

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at org.apache.spark.sql.errors.QueryCompilationErrors$.infiniteStreamingTriggerNotSupportedError(QueryCompilationErrors.scala:7660)
	at org.apache.spark.sql.streaming.DataStreamUtils$.checkTriggerSupportedOnServerless(DataStreamUtils.scala:601)
	at org.apache.spark.sql.classic.streaming.startInternal(DataStreamWriter.scala:523)
	at org.apache.spark.sql.classic.streaming.writeToV1Table$1(DataStreamWriter.scala:330)
	at org.apache.spark.sql.classic.streaming.$anonfun$toTable$1(DataStreamWriter.scala:365)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.sql.classic.streaming.toTable(DataStreamWriter.scala:231)
	at org.apache.spark.sql.classic.streaming.toTable(DataStreamWriter.scala:90)
	at org.apache.spark.sql.connect.

- **Calling `show` or `count` on a streaming DataFrame.** Write the stream to
  a table with `writeStream`, then query the table, as this page does.
- **Grouping by processing time when the question is about event time.** The
  {ref}`out-of-order example <stream-out-of-order>` showed how the two
  disagree.
- **Treating the checkpoint as disposable.** It is what lets the second run
  above skip the rows it already processed. Delete it, and the next run
  starts over from the beginning of the source.

Clean up the table and the volume:

In [8]:
spark.sql("DROP TABLE IF EXISTS workspace.default.stream01_sales_copy")
spark.sql("DROP VOLUME IF EXISTS workspace.default.streaming_01");

Next, {doc}`Spark Structured Streaming <02_spark_structured_streaming>`
explains the programming model behind that query: sources, sinks, output
modes, triggers and checkpoints.

## Resources

- [Structured Streaming Programming Guide (Apache Spark)](https://spark.apache.org/docs/latest/streaming/index.html)
- [Streaming on Databricks (Databricks)](https://docs.databricks.com/aws/en/structured-streaming/)
- [Structured Streaming concepts (Databricks)](https://docs.databricks.com/aws/en/structured-streaming/concepts)